# 05 — Matmul & Aggregations

**Dataset**: `sklearn.datasets.load_digits` — 1797 handwritten digits, 8×8 grayscale images, 10 classes.  
**Goal**: Linear projections over flattened images, pairwise similarity maps via matrix multiplication,  
and batched matrix multiplications (`bmm`) — side-by-side in NumPy and PyTorch.

### Key Distinction
| Function | Constraint |
|---|---|
| `torch.mm` / `np.matmul` (2-D) | strictly 2-D |
| `torch.matmul` / `@` | auto-broadcasts, handles 1-D to N-D |
| `torch.bmm` | explicit batch: `(B,n,m) @ (B,m,p)` |

In [ ]:
# ── Shared Setup ────────────────────────────────────────────────────────────
from sklearn.datasets import load_digits
import numpy as np
import torch
import matplotlib.pyplot as plt

digits = load_digits()

np_images = digits.images.astype(np.float32)   # (1797, 8, 8)
np_labels = digits.target.astype(np.int64)     # (1797,)

# Work with flattened images for matrix ops
np_flat = np_images.reshape(len(np_images), -1)  # (1797, 64)

pt_images = torch.tensor(np_images)
pt_flat   = torch.tensor(np_flat)
pt_labels = torch.tensor(np_labels)

print(f'flat images: {np_flat.shape}')

---
## P1 — Linear Projection: X @ W

Project the 64-d flat images down to 16 dimensions using a random weight matrix.  
This is exactly what `nn.Linear(64, 16, bias=False)` does internally.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
rng = np.random.default_rng(42)
np_W = rng.standard_normal((64, 16)).astype(np.float32) * 0.01   # (64, 16)

np_proj = np_flat @ np_W            # (1797, 64) @ (64, 16) → (1797, 16)
print('projection shape:', np_proj.shape)

#### Drill — `linear_projection`
Practice the core operation before using it in the problem above.

In [ ]:
X = torch.ones(2, 3)
W = torch.ones(3, 4)
# DRILL: matrix multiply X and W
Y = torch.matmul(X, W)
assert Y.shape == (2, 4)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def linear_projection(X_t, W_np):
    """
    Project (N, 64) → (N, 16) using weight matrix W.

    Returns:
        proj_mm     : using torch.mm     (2-D only)
        proj_matmul : using torch.matmul (general)
        proj_at     : using @ operator
    """
    tW = torch.tensor(W_np)

    # torch.mm — 2-D only: (N, 64) @ (64, 16) → (N, 16)
    proj_mm     = ...   # torch.mm(X_t, tW)

    # torch.matmul — same for 2-D, but also handles batched
    proj_matmul = ...   # torch.matmul(X_t, tW)

    # @ operator — syntactic sugar for matmul
    proj_at     = ...   # X_t @ tW

    return proj_mm, proj_matmul, proj_at

t_proj_mm, t_proj_matmul, t_proj_at = linear_projection(pt_flat, np_W)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
for t_proj in [t_proj_mm, t_proj_matmul, t_proj_at]:
    assert t_proj.shape == (1797, 16)
    assert np.allclose(np_proj, t_proj.numpy(), atol=1e-3)
print('P1 assertions passed ✓')

---
## P2 — Pairwise Image Similarity: X @ X^T

Compute the dot-product similarity between every pair of images.  
Result: (N, N) similarity matrix — `sim[i, j]` = dot product of image i and image j.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
# Use first 100 images to keep it tractable
N_sub = 100
np_sub = np_flat[:N_sub]                                # (100, 64)
np_sim = np_sub @ np_sub.T                              # (100, 100)

print('similarity matrix shape:', np_sim.shape)
print('diagonal (self-similarity) sample:', np_sim[0, 0].round(1))

#### Drill — `pairwise_similarity`
Practice the core operation before using it in the problem above.

In [ ]:
A = torch.ones(2, 3)
B = torch.ones(4, 3)
# DRILL: A @ B.T for pairwise similarity
sim = A @ B.T
assert sim.shape == (2, 4)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def pairwise_similarity(X_t, n=100):
    """
    Compute (n, n) dot-product similarity matrix for first n images.
    """
    sub = X_t[:n]                    # (n, 64)

    # X @ X^T
    # .t() is the 2-D transpose shorthand
    sim = ...   # sub @ sub.t()     or  torch.mm(sub, sub.t())

    return sim

t_sim = pairwise_similarity(pt_flat)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_sim.shape == (100, 100)
assert np.allclose(np_sim, t_sim.numpy(), atol=1e-2)
# Diagonal should be non-negative (squared norms)
assert (t_sim.diag() >= -1e-5).all()
print('P2 assertions passed ✓')

# Visualize the similarity matrix
plt.figure(figsize=(6, 5))
plt.imshow(t_sim.numpy(), cmap='hot', interpolation='nearest')
plt.colorbar(label='dot product')
plt.title('Pairwise image similarity (first 100)')
plt.xlabel('image index'); plt.ylabel('image index')
plt.tight_layout(); plt.show()

---
## P3 — Batched Matrix Multiply (bmm)

Batch matrix multiply: treat each image as a row vector `(1, 64)`,  
multiply by a shared projection `(64, 16)`, using `bmm`.

`bmm` requires both inputs to be 3-D: `(B, n, m) × (B, m, p) → (B, n, p)`.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
B = 32
np_batch_X = np_flat[:B].reshape(B, 1, 64)                   # (32, 1, 64)
np_batch_W = np.broadcast_to(np_W[np.newaxis], (B, 64, 16))  # (32, 64, 16)

# Batched matmul: (B, 1, 64) @ (B, 64, 16) → (B, 1, 16)
np_bmm = np.matmul(np_batch_X, np_batch_W)                   # (32, 1, 16)

print('bmm result shape:', np_bmm.shape)

#### Drill — `batched_matmul`
Practice the core operation before using it in the problem above.

In [ ]:
X = torch.ones(2, 5, 3)
W = torch.ones(3, 4)
# DRILL: broadcasted matmul with matrix
Y = X @ W
assert Y.shape == (2, 5, 4)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def batched_matmul(X_t, W_np, batch_size=32):
    """
    Use torch.bmm for batched projection.

    Returns:
        result : (B, 1, 16)
    """
    tW = torch.tensor(W_np)                              # (64, 16)
    tX = X_t[:batch_size].unsqueeze(1)                   # (B, 1, 64)

    # Expand W to batch: (64, 16) → (B, 64, 16)
    tW_batched = ...   # tW.unsqueeze(0).expand(batch_size, -1, -1)

    # torch.bmm: explicit 3-D batch multiply
    # (B, 1, 64) × (B, 64, 16) → (B, 1, 16)
    result = ...       # torch.bmm(tX, tW_batched)

    return result

t_bmm = batched_matmul(pt_flat, np_W)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_bmm.shape == (32, 1, 16)
assert np.allclose(np_bmm, t_bmm.numpy(), atol=1e-3)
# Also verify it matches the simpler 2D projection
simple = pt_flat[:32] @ torch.tensor(np_W)               # (32, 16)
assert np.allclose(simple.numpy(), t_bmm.squeeze(1).numpy(), atol=1e-3)
print('P3 assertions passed ✓')

---
## P4 — Einsum: Flexible Tensor Contraction

`torch.einsum` / `np.einsum` uses Einstein summation notation.  
It's the Swiss Army knife for any contraction, transpose, or trace.

Tasks:
1. Matrix multiply via einsum
2. Batched dot product
3. Outer product

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_sub5   = np_flat[:5]                                # (5, 64)

# 1. Matrix multiply: 'ij,jk->ik'  ≡  X @ W
np_ein_mm = np.einsum('ij,jk->ik', np_sub5, np_W)     # (5, 16)

# 2. Batched dot product: 'bi,bi->b'  ≡  rowwise dot
np_ein_dot = np.einsum('bi,bi->b', np_sub5, np_sub5)  # (5,) — squared norms

# 3. Outer product of two 1-D vectors
a, b = np_flat[0], np_flat[1]                          # each (64,)
np_ein_outer = np.einsum('i,j->ij', a, b)              # (64, 64)

print('einsum mm:', np_ein_mm.shape)
print('einsum dot:', np_ein_dot.shape)
print('einsum outer:', np_ein_outer.shape)

#### Drill — `einsum_ops`
Practice the core operation before using it in the problem above.

In [ ]:
A = torch.ones(2, 3)
B = torch.ones(3, 4)
# DRILL: matrix multiply using einsum
C = torch.einsum('ij,jk->ik', A, B)
assert C.shape == (2, 4)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def einsum_ops(X_t, W_np):
    """
    Returns:
        t_ein_mm    : (5, 16) — matmul via einsum
        t_ein_dot   : (5,)    — batched dot product
        t_ein_outer : (64, 64) — outer product
    """
    tW   = torch.tensor(W_np)
    sub5 = X_t[:5]

    # 1. matmul: 'ij,jk->ik'
    t_ein_mm = ...     # torch.einsum('ij,jk->ik', sub5, tW)

    # 2. batched dot: 'bi,bi->b'
    t_ein_dot = ...    # torch.einsum('bi,bi->b', sub5, sub5)

    # 3. outer: 'i,j->ij'
    a, b = X_t[0], X_t[1]
    t_ein_outer = ...  # torch.einsum('i,j->ij', a, b)

    return t_ein_mm, t_ein_dot, t_ein_outer

t_ein_mm, t_ein_dot, t_ein_outer = einsum_ops(pt_flat, np_W)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.allclose(np_ein_mm,    t_ein_mm.numpy(),    atol=1e-3)
assert np.allclose(np_ein_dot,   t_ein_dot.numpy(),   atol=1e-2)
assert np.allclose(np_ein_outer, t_ein_outer.numpy(), atol=1e-2)
print('P4 assertions passed ✓')

---
## P5 — SVD & PCA on Digit Images

Use SVD to find the principal components of the flat digit images.  
Project onto the top 2 and visualise the class separation.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_f64 = np_flat.astype(np.float64)                          # precision matters for linalg
np_Xc  = np_f64 - np_f64.mean(axis=0)                       # centre the data
np_U, np_S, np_Vt = np.linalg.svd(np_Xc, full_matrices=False)

# Project onto top 2 components
np_pca2 = np_Xc @ np_Vt[:2].T                               # (1797, 2)

print('SVD: U', np_U.shape, 'S', np_S.shape, 'Vt', np_Vt.shape)
print('PCA projection:', np_pca2.shape)

#### Drill — `svd_pca`
Practice the core operation before using it in the problem above.

In [ ]:
X = torch.ones(2, 2)
# DRILL: compute SVD
U, S, V = torch.svd(X)
assert S.numel() == 2

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def svd_pca(X_t):
    """
    Compute SVD of centered data and project onto top 2 PCs.

    Returns:
        t_S    : (64,)      — singular values
        t_pca2 : (1797, 2)  — PCA projection
    """
    tX = X_t.to(torch.float64)                  # use float64 for stability
    t_Xc = tX - tX.mean(dim=0)                  # centre

    # torch.linalg.svd returns (U, S, Vh) where Vh = Vᵀ
    # NumPy equivalent: np.linalg.svd(Xc, full_matrices=False)
    t_U, t_S, t_Vh = ...   # torch.linalg.svd(t_Xc, full_matrices=False)

    # Project onto top 2: Xc @ Vh[:2].T
    t_pca2 = ...           # t_Xc @ t_Vh[:2].T

    return t_S, t_pca2

t_S, t_pca2 = svd_pca(pt_flat)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.allclose(np_S, t_S.numpy(), atol=1e-3)
# PCA signs can flip — compare absolute values
assert np.allclose(np.abs(np_pca2), np.abs(t_pca2.numpy()), atol=1e-2)
print('P5 assertions passed ✓')

# Visualize PCA
pca_np = t_pca2.numpy()
plt.figure(figsize=(8, 6))
for c in range(10):
    mask = np_labels == c
    plt.scatter(pca_np[mask, 0], pca_np[mask, 1], label=str(c), alpha=0.6, s=15)
plt.title('PCA of Digits (top 2 components via SVD)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(title='digit', markerscale=2)
plt.tight_layout(); plt.show()